In [110]:
import sys
import os

print("python:", sys.executable)
print("Exixts:", os.path.exists(sys.executable))

python: c:\Projects\Spark_practice\Spark_course\.venv\Scripts\python.exe
Exixts: True


In [111]:
import sys
import os

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

print(sys.executable)

c:\Projects\Spark_practice\Spark_course\.venv\Scripts\python.exe


In [112]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("RDD Fundamentals")
    .master("local[2]")
    .getOrCreate()
)

sc = spark.sparkContext

print(sc.version)

3.5.6


In [ ]:
#### WIDE TRANSFORMAION ########

# * SET/DATASET
# * Distinct()
# * intersection()
# * subtarct()

In [ ]:
# distinct() : Removes duplicate elemnts from an RDD

# distinct(numPartitions) : The desired number of partition for the result \

In [37]:
rdd = sc.parallelize([
    10,20,30,20,10,40,10,50
])
result = rdd.distinct()
result.collect()

[10, 20, 30, 40, 50]

In [38]:
# Find Unique users

events=sc.parallelize([
    ("U101","Product_1"),
    ("U102","Product_2"),
    ("U101","Product_1"),
    ("U103","Product_3"),
    ("U102","Product_2"),
    ("U102","Product_2"),
    ("U102","Product_2"),
    ("U101","Product_1"),
])

result = events.distinct()
result.collect()



[('U102', 'Product_2'), ('U101', 'Product_1'), ('U103', 'Product_3')]

In [39]:
rdd=sc.parallelize([
    10,15,20,25,30,35
])

# Even No togather and Odd no togather 


In [40]:
result = rdd.groupBy(
    lambda x: "EVEN" if x % 2==0 else "ODD"
)

In [42]:

for key,values in result.collect():
    print(key,tuple(values))

EVEN (10, 20, 30)
ODD (15, 25, 35)


In [43]:
# Example 1- groupBy()

transactions = sc.parallelize([
    ("T001", "C101", 5000, "INDIA"),
    ("T002", "C102", 18000, "USA"),
    ("T003", "C103", 75000, "INDIA"),
    ("T004", "C104", 120000, "UK"),
    ("T005", "C105", 35000, "INDIA"),
    ("T006", "C106", 95000, "USA"),
    ("T007", "C107", 8000, "UK")
], 3)

In [44]:
def get_amount_category(records):
    amount = records[2]

    if amount >=100000:
        return "CRITICAL"
    elif amount >=50000:
        return "HIGH"
    elif amount >=10000:
        return "MEDIUM"
    else:
        return "LOW"

In [45]:
result = transactions.groupBy(get_amount_category)
for category, records in result.collect():
    print("\n",category)
    for record in records:
        print(record)


 HIGH
('T003', 'C103', 75000, 'INDIA')
('T006', 'C106', 95000, 'USA')

 LOW
('T001', 'C101', 5000, 'INDIA')
('T007', 'C107', 8000, 'UK')

 MEDIUM
('T002', 'C102', 18000, 'USA')
('T005', 'C105', 35000, 'INDIA')

 CRITICAL
('T004', 'C104', 120000, 'UK')


In [46]:
# Example :2 
logs = sc.parallelize([
    ("2026-08-18 10:01", "payment", "ERROR", "Database connection timeout"),
    ("2026-08-18 10:02", "order", "ERROR", "API connection timeout"),
    ("2026-08-18 10:03", "payment", "ERROR", "Invalid authentication token"),
    ("2026-08-18 10:04", "customer", "ERROR", "Database connection refused"),
    ("2026-08-18 10:05", "order", "ERROR", "Out of memory"),
    ("2026-08-18 10:06", "payment", "ERROR", "Authentication failed")
], 3)

In [47]:
def error_category(records):
    message = records[3].lower()

    if "database" in message:
        return "DATABASE_ERROR"
    elif "timeout" in message:
        return "Timeout_Error"
    elif "authentication" in message:
        return "AUTH_ERROR"
    elif "memory" in message:
        return "MEMORY_ERROR"
    else:
        return "OTHER_ERROR"


In [48]:
result = logs.groupBy(error_category)

In [49]:
for category,records in result.collect():
    print(category)
    for record in records:
        print(record)

Timeout_Error
('2026-08-18 10:02', 'order', 'ERROR', 'API connection timeout')
AUTH_ERROR
('2026-08-18 10:03', 'payment', 'ERROR', 'Invalid authentication token')
('2026-08-18 10:06', 'payment', 'ERROR', 'Authentication failed')
DATABASE_ERROR
('2026-08-18 10:01', 'payment', 'ERROR', 'Database connection timeout')
('2026-08-18 10:04', 'customer', 'ERROR', 'Database connection refused')
MEMORY_ERROR
('2026-08-18 10:05', 'order', 'ERROR', 'Out of memory')


In [50]:
#Example -1 ReduceByKey()

sales =sc.parallelize([
    ("INDIA",1000),
    ("USA",7000),
    ("INDIA",5000),
    ("US",3000),
    ("UK",9000),
    ("USA",6000),
    ("USA",5000),
    ("INDIA",3000),
    ("UK",2000),
])

# Requirment:

# Calculate total sales for each country

In [51]:
rdd = sales.reduceByKey(lambda x,y:x+y)
rdd.collect()

[('UK', 11000), ('INDIA', 9000), ('USA', 18000), ('US', 3000)]

In [53]:
# example 2: reduceByKey()

transactions = sc.parallelize([
    ("T001", "C101", 5000, "INDIA"),
    ("T002", "C102", 18000, "USA"),
    ("T003", "C103", 75000, "INDIA"),
    ("T004", "C104", 120000, "UK"),
    ("T005", "C105", 35000, "INDIA"),
    ("T006", "C106", 95000, "USA"),
    ("T007", "C107", 8000, "UK")
], 3)

# Requirment: Calculate total transection amount by country

In [54]:
pair_rdd = transactions.map(
    lambda x: (x[3],x[2])
)

In [55]:
def add_values(x,y):
    return x+y

In [56]:
result = pair_rdd.reduceByKey(add_values)
result.collect()

[('UK', 128000), ('INDIA', 115000), ('USA', 113000)]

In [57]:
## ============================================================
# QUESTION 1
# Consolidate Failed Customers Across Two Days
# Level: MEDIUM
# ============================================================

day1 = sc.parallelize([
    ("T001", "C101", 1200, "SUCCESS"),
    ("T002", "C102", 2500, "FAILED"),
    ("T003", "C103", 1800, "FAILED"),
    ("T004", "C104", 3200, "SUCCESS"),
    ("T005", "C105", 900,  "SUCCESS")
])

day2 = sc.parallelize([
    ("T006", "C102", 2100, "FAILED"),
    ("T007", "C104", 4500, "FAILED"),
    ("T008", "C106", 5100, "SUCCESS"),
    ("T009", "C103", 1400, "FAILED"),
    ("T010", "C107", 3000, "FAILED")
])

In [58]:
combine = day1.union(day2)
failed = combine.filter(
    lambda x: x[3] == "FAILED"
)
customer_ids = failed.map(lambda x:x[1])
unique_ids = customer_ids.distinct()
unique_ids.collect()

['C103', 'C104', 'C107', 'C102']

In [59]:
# ============================================================
# QUESTION 2
# Successful Customers Present in Both Months
# Level: MEDIUM
# ============================================================

july = sc.parallelize([
    ("T001", "C101", 1200, "SUCCESS"),
    ("T002", "C102", 3000, "FAILED"),
    ("T003", "C103", 2500, "SUCCESS"),
    ("T004", "C104", 1800, "SUCCESS"),
    ("T005", "C101", 900,  "SUCCESS"),
    ("T006", "C105", 5000, "SUCCESS")
])

august = sc.parallelize([
    ("T101", "C101", 2000, "SUCCESS"),
    ("T102", "C103", 1500, "FAILED"),
    ("T103", "C104", 3500, "SUCCESS"),
    ("T104", "C106", 4200, "SUCCESS"),
    ("T105", "C105", 2200, "SUCCESS"),
    ("T106", "C107", 9000, "SUCCESS")
])


In [60]:

july_new = july.filter(lambda x:x[3] == "SUCCESS").map(lambda x:x[1]).distinct()
august_new = august.filter(lambda x:x[3] == "SUCCESS").map(lambda x:x[1]).distinct()

result = july_new.intersection(august_new)
result.collect()


['C101', 'C104', 'C105']

In [61]:

# ============================================================
# QUESTION 3
# Customers Who Stopped Transacting
# Level: MEDIUM
# ============================================================

july_transactions = sc.parallelize([
    ("T001", "C101", 1000),
    ("T002", "C102", 2000),
    ("T003", "C103", 3000),
    ("T004", "C101", 1500),
    ("T005", "C104", 4000),
    ("T006", "C105", 5000)
])

august_transactions = sc.parallelize([
    ("T101", "C101", 2500),
    ("T102", "C103", 3500),
    ("T103", "C106", 6000),
    ("T104", "C103", 1200),
    ("T105", "C105", 1700)
])

In [62]:
july_new = july_transactions.filter(lambda x: x[2]).map(lambda x:x[1]).distinct()
august_new = august_transactions.filter(lambda x: x[2]).map(lambda x:x[1]).distinct()
result = july_new.subtract(august_new)
result.collect()

['C104', 'C102']

In [30]:
# ============================================================
# QUESTION 4
# High-Value International Spend Per Customer
# Level: MEDIUM-HARD
# ============================================================

transactions = sc.parallelize([
    ("T001", "C101", 12000, "INDIA",   "SUCCESS"),
    ("T002", "C101", 45000, "USA",     "SUCCESS"),
    ("T003", "C102", 28000, "UK",      "SUCCESS"),
    ("T004", "C102", 32000, "USA",     "FAILED"),
    ("T005", "C101", 18000, "UAE",     "SUCCESS"),
    ("T006", "C103", 70000, "INDIA",   "SUCCESS"),
    ("T007", "C103", 25000, "GERMANY", "SUCCESS"),
    ("T008", "C104", 9000,  "USA",     "SUCCESS"),
    ("T009", "C104", 27000, "UK",      "SUCCESS"),
    ("T010", "C105", 56000, "UAE",     "FAILED")
])

In [33]:
success = transactions.filter(lambda x:x[4] == "SUCCESS" and x[3] != "INDIA" and x[2]>10000)
customer_amount = success.map(lambda x: (x[1],x[2]))
total_spend= customer_amount.reduceByKey(lambda a,b:a+b)

for customer,amount in total_spend.collect():
    print(customer, "-->", amount)



C102 --> 28000
C103 --> 25000
C101 --> 63000
C104 --> 27000


In [67]:
# ============================================================
# QUESTION 5
# Customers Crossing Spending Threshold
# Level: MEDIUM-HARD
# ============================================================

transactions = sc.parallelize([
    ("C101", 12000),
    ("C102", 8000),
    ("C101", 18000),
    ("C103", 25000),
    ("C102", 7000),
    ("C101", 22000),
    ("C103", 9000),
    ("C104", 45000),
    ("C104", 10000),
    ("C105", 15000)
])

In [68]:
group_data = transactions.reduceByKey(lambda x,y: x+y)
result = group_data.filter(lambda x:x[1] > 40000)
result.collect()

[('C101', 52000), ('C104', 55000)]

In [70]:
# ============================================================
# QUESTION 7
# Unique Error Applications Across Two Servers
# Level: HARD
# ============================================================

server1_logs = sc.parallelize([
    ("2026-08-18 10:01", "PAYMENT", "ERROR",   "Timeout"),
    ("2026-08-18 10:02", "AUTH",    "INFO",    "Login"),
    ("2026-08-18 10:03", "ORDER",   "ERROR",   "Database unavailable"),
    ("2026-08-18 10:04", "PAYMENT", "ERROR",   "HTTP 500"),
    ("2026-08-18 10:05", "SEARCH",  "WARNING", "Slow response")
])

server2_logs = sc.parallelize([
    ("2026-08-18 10:06", "AUTH",    "ERROR", "Invalid token"),
    ("2026-08-18 10:07", "PAYMENT", "ERROR", "Connection refused"),
    ("2026-08-18 10:08", "ORDER",   "INFO",  "Order created"),
    ("2026-08-18 10:09", "PROFILE", "ERROR", "Service unavailable")
])


In [71]:
combine_logs = server1_logs.union(server2_logs)
filtering = combine_logs.filter(lambda x:x[2] == "ERROR").map(lambda x:x[1]).distinct()
filtering.collect()

['PAYMENT', 'ORDER', 'PROFILE', 'AUTH']

In [72]:
 # ============================================================
# QUESTION 6
# Source vs Target Reconciliation
# Level: HARD
# ============================================================

source = sc.parallelize([
    ("T001", "C101", 1000, "SUCCESS"),
    ("T002", "C102", 2000, "SUCCESS"),
    ("T003", "C103", 3000, "FAILED"),
    ("T004", "C104", 4000, "SUCCESS"),
    ("T005", "C105", 5000, "SUCCESS"),
    ("T006", "C106", 6000, "SUCCESS"),
    ("T007", "C107", 7000, "FAILED")
])

target = sc.parallelize([
    ("T001", "C101", 1000),
    ("T002", "C102", 2000),
    ("T004", "C104", 4000),
    ("T008", "C108", 8000)
])

In [ ]:
comp_source = source.filter(lambda x:x[3] == "SUCCESS").map(lambda x:x[0]).distinct()
comp_target = target.map(lambda x:x[0]).distinct()

matched = comp_source.intersection(comp_target)
missing = comp_source.subtract(comp_target)
extra = comp_target.subtract(comp_source)

print("\n","Matched")
for tid in matched.collect():
    print(tid)

print("\n","Missing")
for tid in missing.collect():
    print(tid)
    
print("\n","Extra")
for tid in extra.collect():
    print(tid)


 Matched
T001
T002
T004

 Missing
T006
T005

 Extra
T008


In [84]:
# ============================================================
# QUESTION 9
# Group Transactions Into Risk Categories
# Level: HARD
# ============================================================

transactions = sc.parallelize([
    ("T001", "C101", 5000,   "INDIA"),
    ("T002", "C102", 25000,  "USA"),
    ("T003", "C103", 75000,  "UK"),
    ("T004", "C104", 110000, "UAE"),
    ("T005", "C105", 45000,  "INDIA"),
    ("T006", "C106", 95000,  "GERMANY"),
    ("T007", "C107", 8000,   "USA"),
    ("T008", "C108", 55000,  "INDIA"),
    ("T009", "C109", 150000, "USA")
])


In [93]:
not_India = transactions.filter(lambda x:x[3] != "INDIA").map(lambda x:(x[0],x[2]))



In [101]:
result = not_India.reduceByKey(lambda x,y:x+y)

def risk_category(records):
    amount = records[1]
    
    if amount < 10000:
        return "Low"
    elif amount < 50000:
        return "mediium"
    elif amount < 100000:
        return "High"
    else:
        return "Critical"



In [105]:
act_result = result.groupBy(risk_category)
for category, values in act_result.collect():
    print(category, list(values))

Critical [('T004', 110000), ('T009', 150000)]
Low [('T007', 8000)]
mediium [('T002', 25000)]
High [('T003', 75000), ('T006', 95000)]


In [107]:
# ============================================================
# QUESTION 8
# Count Errors Per Application Across Multiple Servers
# Level: HARD
# ============================================================

server1_logs = sc.parallelize([
    ("PAYMENT", "ERROR"),
    ("AUTH",    "SUCCESS"),
    ("ORDER",   "ERROR"),
    ("PAYMENT", "ERROR"),
    ("SEARCH",  "SUCCESS")
])

server2_logs = sc.parallelize([
    ("AUTH",    "ERROR"),
    ("PAYMENT", "ERROR"),
    ("ORDER",   "SUCCESS"),
    ("PROFILE", "ERROR"),
    ("PAYMENT", "SUCCESS")
])

server3_logs = sc.parallelize([
    ("PAYMENT", "ERROR"),
    ("AUTH",    "ERROR"),
    ("ORDER",   "ERROR"),
    ("PROFILE", "SUCCESS")
])

In [109]:
combining = server1_logs.union(server2_logs).union(server3_logs)

In [ ]:
filtering = combining.filter(lambda x:x[1] == "ERROR").map(lambda x:(x[0],1))

result = filtering.reduceByKey(lambda x,y:x+y) 

for auth, count in result.collect():
    print(auth, "-->", count)

PAYMENT --> 4
ORDER --> 2
PROFILE --> 1
AUTH --> 2


In [119]:
# ============================================================
# QUESTION 10
# Customer Transaction Summary
# Level: HARD
# ============================================================

transactions = sc.parallelize([
    ("T001", "C101", 10000, "SUCCESS"),
    ("T002", "C102", 8000,  "SUCCESS"),
    ("T003", "C101", 15000, "FAILED"),
    ("T004", "C103", 25000, "SUCCESS"),
    ("T005", "C101", 22000, "SUCCESS"),
    ("T006", "C102", 7000,  "SUCCESS"),
    ("T007", "C103", 9000,  "FAILED"),
    ("T008", "C104", 45000, "SUCCESS"),
    ("T009", "C101", 13000, "SUCCESS"),
    ("T010", "C104", 5000,  "SUCCESS")
])


In [127]:
filtering = transactions.filter(lambda x:x[3] == "SUCCESS")
pair_rdd = filtering.map(lambda x:(x[1],(x[2],1)))
result = pair_rdd.reduceByKey(lambda x,y:(x[0]+y[0],x[1]+y[1]))

for customer,value in result.collect():
    print(customer, (value[0],value[1]))

C102 (15000, 2)
C103 (25000, 1)
C101 (45000, 3)
C104 (50000, 2)


In [128]:
# ============================================================
# QUESTION 11
# Active Fraud-Watch Customers With High Spend
# Level: HARD
# ============================================================

fraud_watchlist = sc.parallelize([
    "C101",
    "C103",
    "C105",
    "C108",
    "C110"
])

transactions = sc.parallelize([
    ("T001", "C101", 15000, "SUCCESS"),
    ("T002", "C102", 50000, "SUCCESS"),
    ("T003", "C101", 30000, "SUCCESS"),
    ("T004", "C103", 12000, "FAILED"),
    ("T005", "C103", 45000, "SUCCESS"),
    ("T006", "C104", 70000, "SUCCESS"),
    ("T007", "C105", 20000, "SUCCESS"),
    ("T008", "C105", 25000, "SUCCESS"),
    ("T009", "C108", 10000, "FAILED"),
    ("T010", "C108", 18000, "SUCCESS")
])

In [134]:
filtering = transactions.filter(lambda x:x[3] == "SUCCESS")
maping = filtering.map(lambda x:(x[1],x[2]))
total = maping.reduceByKey(lambda x,y:x+y)
total_fil = total.filter(lambda x: x[1] >=40000)
result = total_fil.map(lambda x:x[0]).intersection(fraud_watchlist)
result.collect()


['C103', 'C101', 'C105']

In [135]:
# ============================================================
# QUESTION 12
# Find Customers New This Month
# Level: HARD
# ============================================================

july = sc.parallelize([
    ("T001", "C101", 1000),
    ("T002", "C102", 3000),
    ("T003", "C103", 2500),
    ("T004", "C101", 1800),
    ("T005", "C104", 5000)
])

august = sc.parallelize([
    ("T101", "C101", 2000),
    ("T102", "C103", 4000),
    ("T103", "C105", 3500),
    ("T104", "C106", 1000),
    ("T105", "C105", 2500),
    ("T106", "C107", 9000)
])

In [ ]:
july_cust = july.map(lambda x:x[1])
aug_cust = august.map(lambda x:x[1])
result = aug_cust.subtract(july_cust).distinct()
result.collect()

['C107', 'C105', 'C106']

In [139]:
# ============================================================
# QUESTION 13
# Customers Active in Either Month But Not Both
# Level: HARD
# ============================================================

july = sc.parallelize([
    ("C101", 1000),
    ("C102", 2000),
    ("C103", 3000),
    ("C104", 4000),
    ("C101", 5000)
])

august = sc.parallelize([
    ("C101", 2000),
    ("C103", 3500),
    ("C105", 6000),
    ("C106", 8000),
    ("C105", 1200)
])

In [146]:
print(july.getNumPartitions())

2


In [143]:
july_cust = july.map(lambda x:x[0])
aug_cust = august.map(lambda x:x[0])
july_result = july_cust.subtract(aug_cust).distinct()
aug_result = aug_cust.subtract(july_cust).distinct()
result = july_result.union(aug_result)
result.collect()

['C104', 'C102', 'C105', 'C106']

In [147]:
# ============================================================
# QUESTION 14
# Failed Transaction Amount Per Customer
# Level: HARD
# ============================================================

transactions = sc.parallelize([
    ("T001", "C101", 12000, "SUCCESS"),
    ("T002", "C101", 5000,  "FAILED"),
    ("T003", "C102", 9000,  "FAILED"),
    ("T004", "C101", 7000,  "FAILED"),
    ("T005", "C103", 15000, "SUCCESS"),
    ("T006", "C102", 4000,  "FAILED"),
    ("T007", "C104", 20000, "FAILED"),
    ("T008", "C103", 8000,  "FAILED"),
    ("T009", "C104", 10000, "SUCCESS")
])


In [156]:
filtering = transactions.filter(lambda x:x[3] == "FAILED")
pair_rdd = filtering.map(lambda x:(x[1],x[2]))
result = pair_rdd.reduceByKey(lambda x,y:x+y)
result2 = result.filter(lambda x:x[1] >= 10000)
result2.collect()


[('C102', 13000), ('C101', 12000), ('C104', 20000)]

In [6]:
# ============================================================
# QUESTION 15
# Detect Duplicate Transaction IDs
# Level: HARD
# ============================================================

transactions = sc.parallelize([
    ("T001", "C101", 1000),
    ("T002", "C102", 2000),
    ("T003", "C103", 3000),
    ("T001", "C101", 1000),
    ("T004", "C104", 4000),
    ("T002", "C102", 2000),
    ("T005", "C105", 5000),
    ("T001", "C101", 1000)
])


In [9]:
pair_rdd = transactions.map(lambda x:(x[0],1))
result = pair_rdd.reduceByKey(lambda x,y:x+y)
duplicates = result.filter(lambda x:x[1] > 1)
duplicates.collect()


[('T001', 3), ('T002', 2)]

In [10]:
# ============================================================
# QUESTION 16
# Group S3 Files By Year-Month After Filtering
# Level: HARD
# ============================================================

paths = sc.parallelize([
    "s3://company/sales/year=2026/month=08/day=01/file1.parquet",
    "s3://company/sales/year=2026/month=08/day=02/file2.parquet",
    "s3://company/sales/year=2026/month=07/day=31/file3.parquet",
    "s3://company/logs/year=2026/month=08/day=01/log1.json",
    "s3://company/sales/year=2025/month=12/day=01/file4.parquet",
    "s3://company/logs/year=2025/month=12/day=02/log2.json",
    "s3://company/sales/year=2026/month=08/day=03/file5.parquet"
])

In [11]:
filter = paths.filter(lambda x: "/sales/" in x)
sales_data = filter.map(
    lambda x:( x.split("/")[3].replace("year=", "")+ "-" +
    x.split("/")[4].replace("month", ""),
    x.split("/")[-1])
)
grouped = sales_data.groupByKey()

for month,files in grouped.collect():
    print(month)
    for file in files:
        print(file)

sales-year=2026
file1.parquet
file2.parquet
file3.parquet
file5.parquet
sales-year=2025
file4.parquet


In [12]:
# ============================================================
# QUESTION 17
# Find Customers With Both Success and Failure
# Level: HARD
# ============================================================

transactions = sc.parallelize([
    ("T001", "C101", "SUCCESS"),
    ("T002", "C101", "FAILED"),
    ("T003", "C102", "SUCCESS"),
    ("T004", "C103", "FAILED"),
    ("T005", "C103", "FAILED"),
    ("T006", "C104", "SUCCESS"),
    ("T007", "C104", "FAILED"),
    ("T008", "C105", "SUCCESS"),
    ("T009", "C105", "SUCCESS"),
    ("T010", "C106", "FAILED")
])

In [22]:
group_data = transactions.map(lambda x:(x[2],x[1]))
grouped = group_data.groupByKey()
for status,customer in grouped.collect():
    print (
        status,list(customer)
    )

SUCCESS ['C101', 'C102', 'C104', 'C105', 'C105']
FAILED ['C101', 'C103', 'C103', 'C104', 'C106']


In [23]:
# ============================================================
# QUESTION 18
# Transaction Volume By Country
# Level: HARD
# ============================================================

transactions = sc.parallelize([
    ("T001", "C101", 12000, "INDIA",   "SUCCESS"),
    ("T002", "C102", 22000, "USA",     "SUCCESS"),
    ("T003", "C103", 5000,  "INDIA",   "FAILED"),
    ("T004", "C104", 32000, "USA",     "SUCCESS"),
    ("T005", "C105", 45000, "UK",      "SUCCESS"),
    ("T006", "C106", 8000,  "UK",      "SUCCESS"),
    ("T007", "C107", 51000, "USA",     "FAILED"),
    ("T008", "C108", 27000, "GERMANY", "SUCCESS"),
    ("T009", "C109", 18000, "INDIA",   "SUCCESS"),
    ("T010", "C110", 15000, "UK",      "SUCCESS")
])

In [25]:
filtering = transactions.filter(lambda x:x[4] == "SUCCESS")
pair_rdd = filtering.map(lambda x:(x[3],(x[2],1)))
grouped = pair_rdd.reduceByKey(lambda x,y:(x[0]+y[0],x[1]+y[1]))

for customer,value in grouped.collect():
    print(customer,"-->",(value[0],value[1]))

UK --> (68000, 3)
GERMANY --> (27000, 1)
INDIA --> (30000, 2)
USA --> (54000, 2)


In [34]:
# ============================================================
# QUESTION 19
# High-Risk Customers Seen Across Two Systems
# Level: VERY HARD
# ============================================================

banking_system = sc.parallelize([
    ("C101", 120000),
    ("C102", 45000),
    ("C103", 95000),
    ("C104", 150000),
    ("C105", 70000),
    ("C106", 125000)
])

credit_card_system = sc.parallelize([
    ("C101", 40000),
    ("C103", 30000),
    ("C104", 60000),
    ("C105", 50000),
    ("C107", 140000),
    ("C108", 160000)
])


In [36]:
combined = banking_system.union(credit_card_system)

# Calculate total amount for each customer
total_exposure = combined.reduceByKey(lambda x, y: x + y)

# Keep only customers with exposure >= 150000
high_risk = total_exposure.filter(lambda x: x[1] >= 150000)

# Display result
for customer, amount in high_risk.collect():
    print(customer, "->", amount)


C101 -> 160000
C104 -> 210000
C108 -> 160000


In [39]:
errors = [
    ("PAYMENT", "Database timeout"),
    ("ORDER", "API timeout"),
    ("PAYMENT", "Connection refused"),
    ("CUSTOMER", "Invalid token"),
    ("ORDER", "Out of memory"),
    ("PAYMENT", "Authentication failed")
]

rdd = sc.parallelize(errors, 3)

In [41]:
result = rdd.groupByKey()
for orders, messages in result.collect():
    print("\n",orders)
    for message in messages:
        print(message)


 PAYMENT
Database timeout
Connection refused
Authentication failed

 ORDER
API timeout
Out of memory

 CUSTOMER
Invalid token


In [76]:
transactions = sc.parallelize([
    ("TXN001", "C101", "2026-08-19 09:10:00", "INDIA", "CARD", 1200, "SUCCESS"),
    ("TXN002", "C102", "2026-08-19 09:20:00", "INDIA", "UPI", 800, "SUCCESS"),
    ("TXN003", "C101", "2026-08-19 10:15:00", "INDIA", "CARD", 2500, "FAILED"),
    ("TXN004", "C103", "2026-08-19 10:45:00", "USA", "CARD", 3000, "SUCCESS"),
    ("TXN005", "C102", "2026-08-19 11:30:00", "INDIA", "UPI", 1500, "SUCCESS"),
    ("TXN006", "C104", "2026-08-19 12:10:00", "USA", "BANK", 5000, "SUCCESS"),
    ("TXN007", "C101", "2026-08-19 12:40:00", "INDIA", "CARD", 700, "SUCCESS"),
    ("TXN008", "C103", "2026-08-19 13:15:00", "USA", "UPI", 1100, "FAILED"),
    ("TXN009", "C105", "2026-08-19 14:10:00", "UK", "CARD", 4200, "SUCCESS"),
    ("TXN010", "C102", "2026-08-19 14:50:00", "INDIA", "BANK", 900, "FAILED"),
    ("TXN011", "C104", "2026-08-19 15:30:00", "USA", "CARD", 2200, "SUCCESS"),
    ("TXN012", "C105", "2026-08-19 16:20:00", "UK", "CARD", 1800, "SUCCESS")
], 4)

In [63]:
def get_time_zone(record):
    timeStamp = record[2]

    date_part,time_part=timeStamp.split()
    hour=int(time_part.split(":")[0])
    if 5<= hour <11:
        return "MORNING"
    elif 12<= hour <16:
        return "AFTERNOON"
    elif 17<= hour <21:
        return "EVENING"
    else:
        return "NIGHT"


    

In [64]:
grouped=transactions.groupBy(get_time_zone)
for time,timeStamps in grouped.collect():
    print(time)
    for timeStamp in timeStamps:
        print(timeStamp)


NIGHT
('TXN005', 'C102', '2026-08-19 11:30:00', 'INDIA', 'UPI', 1500, 'SUCCESS')
('TXN012', 'C105', '2026-08-19 16:20:00', 'UK', 'CARD', 1800, 'SUCCESS')
MORNING
('TXN001', 'C101', '2026-08-19 09:10:00', 'INDIA', 'CARD', 1200, 'SUCCESS')
('TXN002', 'C102', '2026-08-19 09:20:00', 'INDIA', 'UPI', 800, 'SUCCESS')
('TXN003', 'C101', '2026-08-19 10:15:00', 'INDIA', 'CARD', 2500, 'FAILED')
('TXN004', 'C103', '2026-08-19 10:45:00', 'USA', 'CARD', 3000, 'SUCCESS')
AFTERNOON
('TXN006', 'C104', '2026-08-19 12:10:00', 'USA', 'BANK', 5000, 'SUCCESS')
('TXN007', 'C101', '2026-08-19 12:40:00', 'INDIA', 'CARD', 700, 'SUCCESS')
('TXN008', 'C103', '2026-08-19 13:15:00', 'USA', 'UPI', 1100, 'FAILED')
('TXN009', 'C105', '2026-08-19 14:10:00', 'UK', 'CARD', 4200, 'SUCCESS')
('TXN010', 'C102', '2026-08-19 14:50:00', 'INDIA', 'BANK', 900, 'FAILED')
('TXN011', 'C104', '2026-08-19 15:30:00', 'USA', 'CARD', 2200, 'SUCCESS')


In [73]:
pair = transactions.map(lambda x:(x[1],x[5]))
group_data = pair.groupByKey()
for customer,tarnsaction in group_data.collect():
    print(customer, list(tarnsaction))

C103 [3000, 1100]
C101 [1200, 2500, 700]
C104 [5000, 2200]
C102 [800, 1500, 900]
C105 [4200, 1800]


In [77]:
pair_rdd = transactions.map(lambda x:(x[1],(x[5],1)))
group_data = pair_rdd.reduceByKey(lambda x,y:(x[0]+y[0],x[1]+y[1]))

for customer,value in group_data.collect():
    print(customer,"-->",(value[0],value[1]))

C103 --> (4100, 2)
C101 --> (4400, 3)
C104 --> (7200, 2)
C102 --> (3200, 3)
C105 --> (6000, 2)


In [78]:
##  Aggregate function

transactions = sc.parallelize([
    ("C101", 100),
    ("C102", 200),
    ("C101", 300),
    ("C103", 400),
    ("C102", 500),
    ("C101", 600)
],3)

In [79]:
def seq_function(result_so_far, value):
    return(
        result_so_far[0]+value,
        result_so_far[1]+1
    )

def combine(acc1,acc2):
    total = acc1[0]+acc2[0]
    count = acc1[1]+acc2[1]
    return(total,count)


In [81]:
result = transactions.aggregateByKey(
    (0,0),
    seq_function,
    combine
)
print(sorted(result.collect()))

[('C101', (1000, 3)), ('C102', (700, 2)), ('C103', (400, 1))]


In [82]:
sales=sc.parallelize([
    ("Apple",10),
    ("Banana",20),
    ("Apple",30),
    ("Banana",40),
    ("Apple",50)
])

In [83]:
def add_values(value_So_far,value):
    return value_So_far+value

def combine_value(tot1,tot2):
    return tot1+tot2

result = sales.aggregateByKey(
    0,
    add_values,
    combine_value
)
result.collect()

[('Apple', 90), ('Banana', 60)]

In [91]:
## Q2. Total Salary Per Department


employees = sc.parallelize([
    ("IT", 50000),
    ("HR", 40000),
    ("IT", 60000),
    ("HR", 45000),
    ("Finance", 70000)
], 2)

In [93]:
def add_value(so_far,value):
    return so_far+value

def combine(tot1,tot2):
    return tot1+tot2

result = employees.aggregateByKey(
    0,
    add_value,
    combine
)
result.collect()

[('IT', 110000), ('HR', 85000), ('Finance', 70000)]

In [94]:
## Q3. Total Orders Per Customer

orders = sc.parallelize([
    ("C101", 2),
    ("C102", 3),
    ("C101", 4),
    ("C103", 1),
    ("C102", 5)
], 2)

In [95]:
def add_value(so_far,value):
    return so_far+value

def combine(tot1,tot2):
    return tot1+tot2

result = orders.aggregateByKey(
    0,
    add_value,
    combine
)
result.collect()

[('C102', 8), ('C103', 1), ('C101', 6)]

In [96]:
## Q19. Sum AND Count Website Response Times

logs = sc.parallelize([
    ("login", 100),
    ("search", 200),
    ("login", 150),
    ("search", 300),
    ("login", 250)
], 3)

In [99]:
def add_values(so_far,value):
    return(
        so_far[0]+value,
        so_far[1]+1
    )

def combine(tot1,tot2):
    total=tot1[0]+tot2[0]
    count=tot1[1]+tot2[1]
    return (total,count)

result = logs.aggregateByKey(
    (0,0),
    add_values,
    combine
)    

result.collect()



[('login', (500, 3)), ('search', (500, 2))]

In [113]:

## Q28. Total Credit and Debit Amount

transactions = sc.parallelize([
    ("C101", ("CREDIT", 1000)),
    ("C101", ("DEBIT", 300)),
    ("C102", ("CREDIT", 500)),
    ("C101", ("CREDIT", 700)),
    ("C102", ("DEBIT", 200))
], 3) 

In [115]:
def add_value(so_far,value):
    debit,credit = so_far
    trans_type,amount = value
    if trans_type == "CREDIT":
        credit += amount
    else:
        debit += amount

    return(credit,debit)

def combine(tot1,tot2):
    return (
        tot1[0]+tot2[0],
        tot1[1]+tot2[1]
    )
     

result = transactions.aggregateByKey(
    (0,0),
    add_value,
    combine
)    

result.collect()

[('C102', (500, 200)), ('C101', (1700, 300))]